# Car damage model — YOLO11-seg training (Google Colab)

Classes: scratch, dent, crack, broken_lamp, glass_shatter, flat_tire

**Pehle:** upar menu mein **Runtime → Change runtime type → T4 GPU** select karein, phir
**Runtime → Run all** dabayein. Poora kaam ~1-2 ghante. Aakhir mein model file
`car_damage_yolo11-seg.pt` khud download ho jayegi — usay apne project ke
`ml_models/` folder mein rakh dein.

Dataset: CarDD (Hugging Face copy) — sirf research / non-commercial license.

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics
import os
os.makedirs("/content/car_inspection/training", exist_ok=True)
os.makedirs("/content/car_inspection/ml_models", exist_ok=True)
%cd /content/car_inspection

## 1. Scripts (project ki wohi files)

In [ ]:
%%writefile training/data.yaml
# YOLO11-seg dataset config — car damage (6 classes)
# prepare_cardd.py chalane ke baad ye folders ban jate hain.
path: ../datasets/car_damage
train: images/train
val: images/val
test: images/test

names:
  0: scratch
  1: dent
  2: crack
  3: broken_lamp
  4: glass_shatter
  5: flat_tire

In [ ]:
%%writefile training/prepare_cardd_hf.py
"""
Hugging Face wali CarDD copy (FiftyOne format) download karke YOLO segmentation
format mein badlo. Form bharne ki zarurat nahi.

    https://huggingface.co/datasets/harpreetsahota/CarDD
    (2816 photos, 6211 damage masks — CarDD ka original train split)

Usage:
    python training/prepare_cardd_hf.py --out datasets/car_damage

Is copy mein sirf train split hai, is liye script khud 80% train / 10% val / 10% test
bana deti hai (fixed seed, har dafa same split).

License: CarDD sirf research / non-commercial use ke liye hai (CarDD_license.pdf dekhein).
"""
import argparse
import base64
import io
import json
import random
import time
import urllib.request
import zlib
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np

HF_BASE = "https://huggingface.co/datasets/harpreetsahota/CarDD/resolve/main/"
CLASS_KEYS = ["scratch", "dent", "crack", "broken_lamp", "glass_shatter", "flat_tire"]
CLASS_MAP = {
    "scratch": "scratch", "dent": "dent", "crack": "crack",
    "lamp broken": "broken_lamp", "glass shatter": "glass_shatter", "tire flat": "flat_tire",
}


def fetch(url, dest):
    if dest.exists() and dest.stat().st_size > 0:
        return
    tmp = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(6):  # internet ki chhoti si gadbad par dobara koshish
        try:
            with urllib.request.urlopen(url, timeout=60) as r, open(tmp, "wb") as f:
                f.write(r.read())
            break
        except OSError:
            if attempt == 5:
                raise
            time.sleep(2 ** attempt)
    tmp.replace(dest)


def decode_mask(det):
    """FiftyOne mask: base64 -> zlib -> np.save array (bbox ke andar ka bool mask)."""
    raw = zlib.decompress(base64.b64decode(det["mask"]["$binary"]["base64"]))
    return np.load(io.BytesIO(raw), allow_pickle=False)


def yolo_lines(sample):
    """Har damage ke mask ko polygon(s) mein badlo: 'cls x1 y1 x2 y2 ...' (0-1)."""
    W, H = sample["metadata"]["width"], sample["metadata"]["height"]
    lines = []
    for det in (sample.get("segmentations") or {}).get("detections", []):
        key = CLASS_MAP.get(det["label"].strip().lower())
        if key is None or "mask" not in det:
            continue
        cls = CLASS_KEYS.index(key)
        bx, by, bw, bh = det["bounding_box"]  # normalized [x, y, w, h]
        mask = decode_mask(det).astype(np.uint8)
        mh, mw = mask.shape
        x0, y0 = bx * W, by * H
        sx, sy = (bw * W) / mw, (bh * H) / mh

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for c in contours:
            if cv2.contourArea(c) < 4:
                continue
            c = cv2.approxPolyDP(c, 1.0, True).reshape(-1, 2).astype(float)
            if len(c) < 3:
                continue
            xs = np.clip((x0 + (c[:, 0] + 0.5) * sx) / W, 0, 1)
            ys = np.clip((y0 + (c[:, 1] + 0.5) * sy) / H, 0, 1)
            pts = " ".join(f"{x:.5f} {y:.5f}" for x, y in zip(xs, ys))
            lines.append(f"{cls} {pts}")
    return lines


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--out", default=Path("datasets/car_damage"), type=Path)
    parser.add_argument("--workers", type=int, default=16)
    args = parser.parse_args()
    out = args.out
    raw = out / "_raw"
    raw.mkdir(parents=True, exist_ok=True)

    print("Annotations download ho rahi hain...")
    fetch(HF_BASE + "samples.json", raw / "samples.json")
    fetch(HF_BASE + "CarDD_license.pdf", out / "CarDD_license.pdf")
    samples = json.loads((raw / "samples.json").read_text(encoding="utf-8"))["samples"]

    print(f"{len(samples)} photos download ho rahi hain...")
    img_raw = raw / "images"
    img_raw.mkdir(exist_ok=True)

    def get(s):
        name = Path(s["filepath"]).name
        fetch(HF_BASE + s["filepath"], img_raw / name)

    done = 0
    with ThreadPoolExecutor(args.workers) as pool:
        for _ in pool.map(get, samples):
            done += 1
            if done % 200 == 0:
                print(f"  {done}/{len(samples)}")

    rng = random.Random(42)
    order = list(range(len(samples)))
    rng.shuffle(order)
    n_val = n_test = len(samples) // 10
    split_of = {}
    for rank, i in enumerate(order):
        split_of[i] = "val" if rank < n_val else "test" if rank < n_val + n_test else "train"

    counts = {"train": 0, "val": 0, "test": 0}
    for i, s in enumerate(samples):
        split = split_of[i]
        name = Path(s["filepath"]).name
        (out / "images" / split).mkdir(parents=True, exist_ok=True)
        (out / "labels" / split).mkdir(parents=True, exist_ok=True)
        dest = out / "images" / split / name
        if not dest.exists():
            (img_raw / name).replace(dest)
        (out / "labels" / split / f"{Path(name).stem}.txt").write_text(
            "\n".join(yolo_lines(s)), encoding="utf-8")
        counts[split] += 1

    print(f"Done -> {out.resolve()}  {counts}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile training/train_damage_model.py
"""
YOLO11-seg ko car damage par train karo.

Usage:
    python training/train_damage_model.py                  # default: yolo11s-seg, 100 epochs
    python training/train_damage_model.py --model yolo11m-seg.pt --epochs 150   # zyada accuracy
    python training/train_damage_model.py --device cpu --batch 4                 # GPU na ho

Model sizes: yolo11n-seg (sab se tez) < s < m < l < x (sab se accurate, slow).
GPU na ho to Google Colab (free T4) par chalayein.

Training ke baad best weights khud copy ho jate hain:
    ml_models/car_damage_yolo11-seg.pt   <- app yahin se load karti hai
"""
import argparse
import shutil
from pathlib import Path

import yaml
from ultralytics import YOLO

ROOT = Path(__file__).resolve().parent.parent


def resolve_data_yaml(data):
    """data.yaml ka relative 'path' project folder ke hisaab se absolute karo."""
    data = Path(data).resolve()
    cfg = yaml.safe_load(data.read_text(encoding="utf-8"))
    if not Path(cfg["path"]).is_absolute():
        cfg["path"] = str((data.parent / cfg["path"]).resolve())
    out = ROOT / "runs" / "data_resolved.yaml"
    out.parent.mkdir(exist_ok=True)
    out.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    return str(out)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", default="yolo11s-seg.pt", help="pretrained COCO weights se start")
    parser.add_argument("--data", default=str(ROOT / "training" / "data.yaml"))
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument("--imgsz", type=int, default=640)
    parser.add_argument("--batch", type=int, default=16)
    parser.add_argument("--device", default=None, help="0 (GPU) ya cpu")
    parser.add_argument("--workers", type=int, default=4, help="data loading threads")
    parser.add_argument("--patience", type=int, default=25)
    parser.add_argument("--resume", action="store_true", help="ruki hui training wahin se chalao")
    args = parser.parse_args()
    data = resolve_data_yaml(args.data)

    if args.resume:
        model = YOLO(str(ROOT / "runs" / "car_damage" / "weights" / "last.pt"))
        model.train(resume=True)
    else:
        model = YOLO(args.model)
        model.train(
            data=data,
            epochs=args.epochs,
            imgsz=args.imgsz,
            batch=args.batch,
            device=args.device,
            patience=args.patience,  # itne epochs tak behtari na ho to ruk jao
            workers=args.workers,
            project=str(ROOT / "runs"),
            name="car_damage",
            exist_ok=True,
            # Car photos ke liye augmentation: ulta (upside down) nahi, left-right theek hai
            fliplr=0.5,
            flipud=0.0,
            degrees=5.0,
            hsv_v=0.4,             # roshni ka farq
        )

    best = ROOT / "runs" / "car_damage" / "weights" / "best.pt"
    model = YOLO(str(best))
    metrics = model.val(data=data, split="test")
    print(f"Test mask mAP50: {metrics.seg.map50:.3f}   mAP50-95: {metrics.seg.map:.3f}")
    print(f"Test box  mAP50: {metrics.box.map50:.3f}   mAP50-95: {metrics.box.map:.3f}")

    target = ROOT / "ml_models" / "car_damage_yolo11-seg.pt"
    target.parent.mkdir(exist_ok=True)
    shutil.copy2(best, target)
    print(f"Model saved -> {target}")


if __name__ == "__main__":
    main()

## 2. Dataset download + YOLO format (~5-10 min)

In [ ]:
!python training/prepare_cardd_hf.py --out datasets/car_damage --workers 16

## 3. Training (T4 GPU par ~1-1.5 ghanta)

`yolo11s-seg` = tez + achhi accuracy. Zyada accuracy chahiye to `yolo11m-seg.pt` likh dein (~2x waqt).
Colab beech mein disconnect ho jaye to neeche wala **resume** cell chalayein.

In [ ]:
!python training/train_damage_model.py --model yolo11s-seg.pt --imgsz 640 --batch 16 --epochs 80 --patience 20 --device 0 --workers 2

In [ ]:
# Sirf tab chalayein jab training beech mein ruk gayi ho
# !python training/train_damage_model.py --resume --device 0

## 4. Results — test set (281 photos jo model ne kabhi nahi dekhi)

In [ ]:
from ultralytics import YOLO
m = YOLO("ml_models/car_damage_yolo11-seg.pt")
r = m.val(data="runs/data_resolved.yaml", split="test", plots=True)
print(f"MASK mAP50 = {r.seg.map50:.3f}   mAP50-95 = {r.seg.map:.3f}")
print(f"BOX  mAP50 = {r.box.map50:.3f}   mAP50-95 = {r.box.map:.3f}")
names = m.names
for i, c in enumerate(r.seg.ap_class_index):
    print(f"  {names[int(c)]:15s} mask mAP50 = {r.seg.ap50[i]:.3f}")

In [ ]:
# Kuch test photos par prediction dekhein
import glob
from IPython.display import Image, display
files = sorted(glob.glob("datasets/car_damage/images/test/*.jpg"))[:6]
for res in m.predict(files, conf=0.35, retina_masks=True, save=True, project="runs", name="preview", exist_ok=True, verbose=False):
    pass
for f in sorted(glob.glob("runs/preview/*.jpg"))[:6]:
    display(Image(f, width=600))

## 5. Model download karein → project ke `ml_models/` folder mein rakhein

In [ ]:
from google.colab import files
files.download("ml_models/car_damage_yolo11-seg.pt")